# Will Betts - data exploration

Cleaned Comtrade exports (`data/cleaned/<year>clean.csv`, 2019-2023) and World Bank GDP (`data/archive/2024-fall/GDP_2023.csv`).

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import linalg as LA
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.linear_model import LinearRegression

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)
sns.set_theme(style="whitegrid")

In [ ]:
REPO = Path.cwd().parent  # notebook lives in willbetts/
DATA = REPO / "data"
YEARS = range(2019, 2024)

# Cleaned exports by country x 18 industry groups (header "Precious Stones\t" has a trailing tab)
exports = {}
for y in YEARS:
    df = pd.read_csv(DATA / "cleaned" / f"{y}clean.csv")
    df.columns = df.columns.str.strip()
    exports[y] = df.set_index("Country")

# World Bank GDP, current US$, 1990-2023 (file starts with a BOM)
gdp = pd.read_csv(DATA / "archive" / "2024-fall" / "GDP_2023.csv", encoding="utf-8-sig")

exports[2023].head()

## Does export diversification predict GDP growth?

1. Each country's export shares across the 18 industry groups (2019).
2. Diversification scores: entropy (higher = more diversified) and HHI (higher = more concentrated).
3. Merge with GDP and compute annualized growth 2019-2023.
4. Plot and regress growth on diversification, controlling for country size.

Diversification is measured in 2019 so it comes *before* the growth it is predicting.

In [ ]:
# 1. Export shares: each row sums to 1
base = exports[2019]
shares = base.div(base.sum(axis=1), axis=0)

# 2. Diversification scores
entropy = -(shares * np.log(shares.where(shares > 0))).sum(axis=1)  # max = log(18) ~ 2.89
hhi = (shares ** 2).sum(axis=1)                                      # min = 1/18 ~ 0.056

div = pd.DataFrame({
    "entropy": entropy,
    "hhi": hhi,
    "top_industry": shares.idxmax(axis=1),
    "top_share": shares.max(axis=1),
    "total_exports": base.sum(axis=1),
})
div.sort_values("entropy").head(10)  # least diversified

In [ ]:
# 3. Comtrade and World Bank spell countries differently; map Comtrade -> World Bank
NAME_MAP = {
    "Bahamas": "Bahamas, The",
    "Bosnia Herzegovina": "Bosnia and Herzegovina",
    "China, Hong Kong Special Administrative Region": "Hong Kong SAR, China",
    "China, Macao Special Administrative Region": "Macao SAR, China",
    "Côte d'Ivoire": "Cote d'Ivoire",
    "Democratic Republic of the Congo": "Congo, Dem. Rep.",
    "Congo": "Congo, Rep.",
    "Egypt": "Egypt, Arab Rep.",
    "France, Monaco": "France",
    "Gambia": "Gambia, The",
    "Kyrgyzstan": "Kyrgyz Republic",
    "Lao People's Dem. Rep.": "Lao PDR",
    "Norway, Svalbard and Jan Mayen": "Norway",
    "Plurinational State of Bolivia": "Bolivia",
    "Republic of Korea": "Korea, Rep.",
    "Republic of Moldova": "Moldova",
    "Saint Lucia": "St. Lucia",
    "Saint Vincent and the Grenadines": "St. Vincent and the Grenadines",
    "Slovakia": "Slovak Republic",
    "State of Palestine": "West Bank and Gaza",
    "Swaziland": "Eswatini",
    "Switzerland, Liechtenstein": "Switzerland",
    "The Former Yugoslav Republic of Macedonia": "North Macedonia",
    "Turkey": "Turkiye",
    "USA, Puerto Rico and US Virgin Islands": "United States",
    "United Republic of Tanzania": "Tanzania",
}
# Expected to drop: EU-28, "Other Asia, nes", Montserrat (no World Bank row);
# Yemen, Lebanon, French Polynesia (no 2023 GDP); United States (missing from GDP_2023.csv)

g = gdp.set_index("Country Name")[["2019", "2023"]].rename(columns=lambda c: f"gdp_{c}")
df = div.rename(index=lambda c: NAME_MAP.get(c, c)).join(g, how="inner").dropna()

df["growth"] = (df["gdp_2023"] / df["gdp_2019"]) ** (1 / 4) - 1  # annualized, nominal US$
df["log_gdp"] = np.log(df["gdp_2019"])

print(f"{len(df)} of {len(div)} countries matched")
unmatched = sorted(set(div.rename(index=lambda c: NAME_MAP.get(c, c)).index) - set(df.index))
print("Unmatched:", unmatched)
df.describe()

In [ ]:
# 4a. Plot: diversification vs growth, bubble size = GDP
fig = px.scatter(
    df.reset_index(names="country"),
    x="entropy", y="growth", size="gdp_2019", color="top_share",
    hover_name="country", hover_data=["top_industry"],
    trendline="ols", size_max=50,
    labels={"entropy": "Export diversification, 2019 (entropy)",
            "growth": "Annualized GDP growth 2019-2023 (nominal US$)",
            "top_share": "Largest industry share"},
)
fig.update_layout(yaxis_tickformat=".0%")
fig.show()

In [ ]:
# 4b. Regressions. Model 2 controls for size (big economies are both more diversified and slower-growing)
m1 = smf.ols("growth ~ entropy", data=df).fit(cov_type="HC1")
m2 = smf.ols("growth ~ entropy + log_gdp", data=df).fit(cov_type="HC1")
print(m1.summary().tables[1])
print(m2.summary().tables[1])
print(f"R^2: {m1.rsquared:.3f} vs {m2.rsquared:.3f}")

In [ ]:
# Outliers worth a look: growth is in nominal US$, so inflation and exchange-rate swings show up here
df.sort_values("growth")[["top_industry", "top_share", "entropy", "growth"]].iloc[np.r_[0:5, -5:0]]

### Things to try next
- Swap `entropy` for `hhi` (expect the opposite sign).
- Drop countries whose top industry is `Energy Drilling and Mining`: the 2019-2023 oil price swing drives their growth.
- Use 2023 diversification, or the *change* in entropy 2019-2023, as the predictor.
- Repeat with a different window (e.g. 2019-2021 for the COVID shock).